# Final Project Phase 2 Summary
This Jupyter Notebook (.ipynb) will serve as the skeleton file for your submission for Phase 2 of the Final Project. Answer all statements addressed below as specified in the instructions for the project, covering all necessary details. Please be clear and concise in your answers. Each response should be at most 3 sentences. Good luck! <br><br>

Note: To edit a Markdown cell, double-click on its text.

## Jupyter Notebook Quick Tips
Here are some quick formatting tips to get you started with Jupyter Notebooks. This is by no means exhaustive, and there are plenty of articles to highlight other things that can be done. We recommend using HTML syntax for Markdown but there is also Markdown syntax that is more streamlined and might be preferable. 
<a href = "https://towardsdatascience.com/markdown-cells-jupyter-notebook-d3bea8416671">Here's an article</a> that goes into more detail. (Double-click on cell to see syntax)

# Heading 1
## Heading 2
### Heading 3
#### Heading 4
<br>
<b>BoldText</b> or <i>ItalicText</i>
<br> <br>
Math Formulas: $x^2 + y^2 = 1$
<br> <br>
Line Breaks are done using br enclosed in < >.
<br><br>
Hyperlinks are done with: <a> https://www.google.com </a> or 
<a href="http://www.google.com">Google</a><br>

# Data Collection and Cleaning
You are required to provide data collection and cleaning for the three (3) minimum datasets. Create a function for each of the following sections that reads or scrapes data from a file or website, manipulate and cleans the parsed data, and writes the cleaned data into a new file. 

Make sure your data cleaning and manipulation process is not too simple. Performing complex manipulation and using modules not taught in class shows effort, which will increase the chance of receiving full credit.


## Data Sources
Include sources (as links) to your datasets. Add any additional data sources if needed. Clearly indicate if a data source is different from one submitted in your Phase I, as we will check that it satisfies the requirements.
*   Downloaded Dataset Source: https://github.com/stabweee/FlightDelay/blob/main/data/interim/ATL_combined.csv
*   ^ Downloaded From: https://www.transtats.bts.gov/ONTIME/Departures.aspx
*   Web Collection #1 (JSON/API) Source: https://open-meteo.com/en/docs/historical-weather-api
*   Web Collection #2 (HTML/Webscrape) Source: https://en.wikipedia.org/wiki/Federal_holidays_in_the_United_States
*   Additional Dataset: https://en.wikipedia.org/wiki/Hartsfield%E2%80%93Jackson_Atlanta_International_Airport#Airline_market_share




In [22]:
import numpy as np
import pandas as pd
from pathlib import Path
import requests
from bs4 import BeautifulSoup

## Downloaded Dataset Requirement

Fill in the predefined functions with your data scraping/parsing code. You may modify/rename each function as you seem fit, but you must provide at least 3 separate functions that clean each of your required datasets.


In [23]:
def downloaded_dataset_parser():
  # Combine separate CSV files into a single DataFrame
  dir_path = Path("../data/raw")
  file_paths = list(dir_path.glob("*.csv"))
  flight_data_separated = [pd.read_csv(file_path)[:-1] for file_path in file_paths]
  flight_data = pd.concat(flight_data_separated, ignore_index=True)

  # Convert format and datatype of date column for sorting and to use as an index with other datasets
  ## notes on format: key + examples: %Y (2026), %y (26), %m (11), %b (Nov), %B (November), %d (03)
  flight_data["Date (MM/DD/YYYY)"] = pd.to_datetime(flight_data["Date (MM/DD/YYYY)"], format="%m/%d/%Y")
  ## Rename column because format in column name no longer accurate
  flight_data.rename(columns={"Date (MM/DD/YYYY)": "Date"}, inplace=True)
  ## Create departure datetime column to compare with weather data
  ### We need to convert datetime to string for the purpose of combining the date with the time to later get the datetime object
  flight_data["Scheduled departure datetime"] = pd.to_datetime(flight_data["Date"].dt.strftime("%Y-%m-%d") + " " + flight_data["Scheduled departure time"])
  ### Round to nearest hour because dataset from API has data recorded at every hour of every day of 2025 
  flight_data["Scheduled departure hour"] = flight_data["Scheduled departure datetime"].dt.floor("h")

  # Carrier Code MQ is replaced with AA because we designate American Eagle as American Airlines
  ## This is to accurately match delays with market share since MQ == AA in terms of carriers listed in ATL market
  flight_data["Carrier Code"] = flight_data["Carrier Code"].replace("MQ", "AA")

  # DataFrame to CSV
  flight_data.to_csv("../data/processed/flights.csv", index=False)

  return flight_data



############ Function Call ############
downloaded_dataset_parser()

,Carrier Code,Date,Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),Scheduled departure datetime,Scheduled departure hour
0,AS,2025-01-01,321.0,N428AS,SEA,18:39,18:36,343.0,344.0,-3.0,18:53,17.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 18:39:00,2025-01-01 18:00:00
1,AS,2025-01-01,355.0,N588AS,SAN,17:19,17:16,284.0,297.0,-3.0,17:35,19.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 17:19:00,2025-01-01 17:00:00
2,AS,2025-01-01,377.0,N278AK,SEA,07:04,06:55,343.0,336.0,-9.0,07:11,16.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 07:04:00,2025-01-01 07:00:00
3,AS,2025-01-01,497.0,N562AS,SEA,15:27,15:22,344.0,338.0,-5.0,15:34,12.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 15:27:00,2025-01-01 15:00:00
4,AS,2025-01-01,783.0,N263AK,PDX,19:59,20:07,319.0,343.0,8.0,20:23,16.0,8.0,0.0,24.0,0.0,0.0,2025-01-01 19:59:00,2025-01-01 19:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301124,UA,2025-12-31,632.0,N18243,IAD,09:40,09:41,110.0,107.0,1.0,09:58,17.0,0.0,0.0,0.0,0.0,0.0,2025-12-31 09:40:00,2025-12-31 09:00:00
301125,UA,2025-12-31,1057.0,N77598,IAH,07:10,07:09,151.0,121.0,-1.0,07:24,15.0,0.0,0.0,0.0,0.0,0.0,2025-12-31 07:10:00,2025-12-31 07:00:00
301126,UA,2025-12-31,1259.0,N76288,SFO,08:25,08:22,335.0,367.0,-3.0,09:31,69.0,0.0,0.0,29.0,0.0,0.0,2025-12-31 08:25:00,2025-12-31 08:00:00
301127,UA,2025-12-31,1831.0,N13716,IAD,14:40,14:37,110.0,105.0,-3.0,14:53,16.0,0.0,0.0,0.0,0.0,0.0,2025-12-31 14:40:00,2025-12-31 14:00:00


## JSON/API Requirement


In [24]:
def json_api_parser():
  # API Call Parameters
  ## lat and lon match with coordinates of Hartsfield-Jackson Airport
  lat = 33.6367
  lon = -84.4279
  ## we are only analyzing the departure delays frrom the year 2025
  min_date = "2025-01-01"
  max_date = "2025-12-31"
  url = "https://archive-api.open-meteo.com/v1/archive"

  params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": min_date,
    "end_date": max_date,
    "temperature_unit": "fahrenheit",
    "hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "cloud_cover", "windspeed_10m", "windgusts_10m", "winddirection_10m", "weathercode"],
  }

  # API Call
  weather = requests.get(url, params=params).json()

  # JSON to DataFrame
  weather_df = pd.DataFrame(weather["hourly"])
  weather_df["time"] = pd.to_datetime(weather_df["time"])

  # DataFrame to CSV
  weather_df.to_csv("../data/processed/weather.csv", index=False)

  return weather_df





############ Function Call ############
json_api_parser()

,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,cloud_cover,windspeed_10m,windgusts_10m,winddirection_10m,weathercode
0,2025-01-01 00:00:00,53.1,59,39.3,45.4,0.0,0.0,0.0,0,19.5,44.6,296,0
1,2025-01-01 01:00:00,52.5,56,37.1,44.5,0.0,0.0,0.0,0,19.3,41.4,302,0
2,2025-01-01 02:00:00,50.3,60,36.8,42.1,0.0,0.0,0.0,0,19.4,41.0,303,0
3,2025-01-01 03:00:00,47.8,65,36.8,39.7,0.0,0.0,0.0,0,19.3,41.4,306,0
4,2025-01-01 04:00:00,46.1,69,36.5,38.2,0.0,0.0,0.0,1,18.6,41.0,310,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00,50.0,18,8.4,40.6,0.0,0.0,0.0,18,12.6,32.4,274,0
8756,2025-12-31 20:00:00,51.0,18,9.3,41.1,0.0,0.0,0.0,1,14.2,34.9,274,0
8757,2025-12-31 21:00:00,50.9,19,10.5,40.9,0.0,0.0,0.0,27,14.6,34.9,265,1
8758,2025-12-31 22:00:00,49.4,21,11.4,39.8,0.0,0.0,0.0,5,13.4,34.2,265,0


## HTML Requirement

In [25]:
def html_parser():
  url = "https://en.wikipedia.org/wiki/Federal_holidays_in_the_United_States"
  agent = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36"}
  response = requests.get(url, headers=agent)
  soup = BeautifulSoup(response.content, "html.parser")

  # Webscrape/extract holiday table
  table = soup.find("table", attrs={"class": "wikitable"})
  holidays = []
  for row in table.find_all("tr")[1:]:
    date = row.find("th").text.strip()
    name = row.find("td").text.strip()
    holidays.append({"Holiday": name, "Date": date})

  # List to DataFrame
  holidays_df = pd.DataFrame(holidays)

  # Convert Date in words to Date Range
  ## We only analyze 2025 dates. We assume that that the accetable range is a maximum of 6 days before and after a holiday
  ## date for the departure to be reasonably associated with the holiday.
  year = 2025
  hdRange = 6
  ## Split strings like "January 15–21(Third Monday)" into month, start day, and end day 
  ### where the end day is NaN for single dates
  #### hyphen not required because of single dates, so we need a noncapturing group  
  dateParts = holidays_df["Date"].str.extract(r"([A-Za-z]+)\s+(\d+)(?:-(\d+))?")
  dateParts.columns = ["Month", "Start Day", "End Day"]
  ## format %B because month given in full proper noun i.e. January vs Jan
  month = pd.to_datetime(dateParts["Month"], format="%B").dt.month
  ## already a range is True for those in dateParts with an End Day
  alrRange = dateParts["End Day"].notna()
  ### If the holiday was provided as a single date, then the start day is the end day
  endDay = dateParts["End Day"].fillna(dateParts["Start Day"])
  ## create start and end days as datetime objects
  start = pd.to_datetime(pd.DataFrame({"year": year, "month": month, "day": dateParts["Start Day"].astype(int)}))
  end = pd.to_datetime(pd.DataFrame({"year": year, "month": month, "day": endDay.astype(int)}))
  ## find the dates that start and end the range of each "holiday season"
  ### .where(cond, thing to swap with if cond false)
  ### TimeDelta basically lets me add or subtract x amount of time from a datetime object
  holidays_df["Start"] = start.where(alrRange, start - pd.Timedelta(days=hdRange))
  holidays_df["End"] = end.where(alrRange, end + pd.Timedelta(days=hdRange))
 #holidays_df = holidays_df.drop(columns="Date")
  ## Date column has been reformatted and renamed 
  holidays_df["Date"] = holidays_df["Start"].astype(str)+ " - " + holidays_df["End"].astype(str)
  holidays_df.rename(columns={"Date" : "Date Range"}, inplace = True)

  # DataFrame to CSV
  holidays_df.to_csv("../data/processed/holidays.csv", index=False)

  return holidays_df

############ Function Call ############
html_parser()

,Holiday,Date Range,Start,End
0,New Year's Day,2024-12-26 - 2025-01-07,2024-12-26,2025-01-07
1,"Birthday of Martin Luther King, Jr.",2025-01-09 - 2025-01-21,2025-01-09,2025-01-21
2,Washington's Birthday,2025-02-09 - 2025-02-21,2025-02-09,2025-02-21
3,Memorial Day,2025-05-19 - 2025-05-31,2025-05-19,2025-05-31
4,Juneteenth National Independence Day,2025-06-13 - 2025-06-25,2025-06-13,2025-06-25
5,Independence Day,2025-06-28 - 2025-07-10,2025-06-28,2025-07-10
6,Labor Day,2025-08-26 - 2025-09-07,2025-08-26,2025-09-07
7,Columbus Day,2025-10-02 - 2025-10-14,2025-10-02,2025-10-14
8,Veterans Day,2025-11-05 - 2025-11-17,2025-11-05,2025-11-17
9,Thanksgiving Day,2025-11-16 - 2025-11-28,2025-11-16,2025-11-28


In [26]:
### Function to add boolean column/series into flight_data so we can easily analyze and compare dates which are now
### assigned True for being within any holiday season
def isHoliday():
  flight_data = pd.read_csv("../data/processed/flights.csv")
  holidays_df = pd.read_csv("../data/processed/holidays.csv")
  flight_data["Date"] = pd.to_datetime(flight_data["Date"])
  holidays_df["Start"] = pd.to_datetime(holidays_df["Start"])
  holidays_df["End"] = pd.to_datetime(holidays_df["End"])
  #### if the Date in flight data is between any holiday season range across all the seasons in holidays, then we can say True
  #### for the Date being within a holiday season
  flight_data["Is Holiday Season"] = holidays_df.apply(lambda row: flight_data["Date"].between(row["Start"], row["End"]), axis=1).any(axis=0)
  flight_data.to_csv("../data/processed/flights.csv", index=False)
  return flight_data

isHoliday()

,Carrier Code,Date,Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),Scheduled departure datetime,Scheduled departure hour,Is Holiday Season
0,AS,2025-01-01,321.0,N428AS,SEA,18:39,18:36,343.0,344.0,-3.0,18:53,17.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 18:39:00,2025-01-01 18:00:00,True
1,AS,2025-01-01,355.0,N588AS,SAN,17:19,17:16,284.0,297.0,-3.0,17:35,19.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 17:19:00,2025-01-01 17:00:00,True
2,AS,2025-01-01,377.0,N278AK,SEA,07:04,06:55,343.0,336.0,-9.0,07:11,16.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 07:04:00,2025-01-01 07:00:00,True
3,AS,2025-01-01,497.0,N562AS,SEA,15:27,15:22,344.0,338.0,-5.0,15:34,12.0,0.0,0.0,0.0,0.0,0.0,2025-01-01 15:27:00,2025-01-01 15:00:00,True
4,AS,2025-01-01,783.0,N263AK,PDX,19:59,20:07,319.0,343.0,8.0,20:23,16.0,8.0,0.0,24.0,0.0,0.0,2025-01-01 19:59:00,2025-01-01 19:00:00,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301124,UA,2025-12-31,632.0,N18243,IAD,09:40,09:41,110.0,107.0,1.0,09:58,17.0,0.0,0.0,0.0,0.0,0.0,2025-12-31 09:40:00,2025-12-31 09:00:00,True
301125,UA,2025-12-31,1057.0,N77598,IAH,07:10,07:09,151.0,121.0,-1.0,07:24,15.0,0.0,0.0,0.0,0.0,0.0,2025-12-31 07:10:00,2025-12-31 07:00:00,True
301126,UA,2025-12-31,1259.0,N76288,SFO,08:25,08:22,335.0,367.0,-3.0,09:31,69.0,0.0,0.0,29.0,0.0,0.0,2025-12-31 08:25:00,2025-12-31 08:00:00,True
301127,UA,2025-12-31,1831.0,N13716,IAD,14:40,14:37,110.0,105.0,-3.0,14:53,16.0,0.0,0.0,0.0,0.0,0.0,2025-12-31 14:40:00,2025-12-31 14:00:00,True


## Additional Dataset Parsing/Cleaning Functions

Write any supplemental (optional) functions here.

In [27]:
airline_codes = {"Delta Air Lines" : "DL", "Frontier Airlines" : "F9", "Southwest Airlines" : "WN", "American Airlines" : "AA", "United Airlines" : "UA", "Spirit Airlines" : "NK", "JetBlue" : "B6", "Alaska Airlines" : "AS", "Air France" : "AF", "Virgin Atlantic" : "VS"}
def extra_source1():
    url = "https://en.wikipedia.org/wiki/Hartsfield-Jackson_Atlanta_International_Airport"
    agent = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36"}
    response = requests.get(url, headers=agent)
    soup = BeautifulSoup(response.content, "html.parser")

    market_shares = []
    section = soup.find("section", attrs={"aria-labelledby": "Airline_market_share"})
    table = section.find("table")
    for row in table.find_all("tr")[1:]:
        entries = [cell.text.strip() for cell in row.find_all("td")]
        market_shares.append(entries)

    # List to DataFrame
    market_shares_df = pd.DataFrame(market_shares, columns=["Rank", "Airline", "Passengers", "Share"])
    
    # Working with dataframe
    ## Remove info kept in () in name of the airline * only applies to Delta and American
    market_shares_df["Airline"] = market_shares_df["Airline"].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
    ## Add columnn matching airline to its corresponding airline code.
    market_shares_df["Carrier Code"] = market_shares_df["Airline"].map(airline_codes)
    ## Re-order the columns so that carrier code is right after Airline
    market_shares_df = market_shares_df[["Rank", "Airline", "Carrier Code", "Passengers", "Share"]]
    ## Drop Other airlines because we do not have any other carriers in the csv file of departure delays.
    ### Value of Other's Carrier Code is NaN.
    market_shares_df = market_shares_df.dropna(subset = ["Carrier Code"])
    ## Convert string of market share percentage into float value for analysis. 
    market_shares_df["Share"] = np.round(market_shares_df["Share"].str.rstrip("%").astype(float) / 100, 4)
    ## We set the Rank column as the index because the rank already orders the airlines by the share.
    market_shares_df = market_shares_df.set_index("Rank")

    # DataFrame to CSV
    market_shares_df.to_csv("../data/processed/market_shares.csv", index=False)

    return market_shares_df

    
############ Function Call ############
extra_source1()

,Airline,Carrier Code,Passengers,Share
Rank,,,,
1,Delta Air Lines,DL,"84,851,005",0.8016
2,Frontier Airlines,F9,"7,053,413",0.0666
3,Southwest Airlines,WN,"3,944,112",0.0373
4,American Airlines,AA,"2,653,658",0.0251
5,United Airlines,UA,"1,937,230",0.0183
6,Spirit Airlines,NK,"1,126,697",0.0106
7,JetBlue,B6,"670,622",0.0063
8,Alaska Airlines,AS,"406,935",0.0038
9,Air France,AF,"369,789",0.0035


In [28]:
# Define further extra source functions as necessary

# Inconsistencies

For each inconsistency (NaN, null, duplicate values, empty strings, etc.) you discover in your datasets, write at least 2 sentences stating the significance, how you identified it, and how you handled it.

1. The flight data stores dates as strings like "01/01/2025" with the scheduled departure time in a separate column while the weather API uses the format of timestamps as "2025-01-01T00:00", and this is significant because the two datasets cannot be sorted chronologically or matched on date and hour without a common format. We identified it by visually comparing the date formats of the two datasets. We handled it by using pd.to_datetime to convert the dates to YYYY-MM-DD and to combine them with the departure time so that each flight's departure can be matched to the hourly weather data. **CSV / Downloaded File**

2. After we scraped and returned the table on holidays, we saw that the "Date" values are text like "January 1 (Fixed Date)", and some entries cover a range of days, so they cannot be compared with the flight dates. This is significant because our goal is to determine whether each flight occurred during a holiday season, and this requires real date values to match against the flight data. We identified it by inspecting the scraped table after parsing it with BeautifulSoup, and we handled it by removing the extra text and converting each entry to a datetime range with a start and an end date with Pandas so it can be matched to the flight dates in a holiday season. **HTML (Public Holidays wiki page)**

3. The "Share" values were scraped as strings like "80.16%," and this is significant because the percent signs make the numbers into strings that cannot be used in calculations, so we need numeric shares to compare each airline's share of delays to its market share. We identified this by viewing the scraped DataFrame and seeing these columns stored as strings. We handled it by stripping the "%" from the strings, then converting and rounding the shares to 4 decimals (e.g., "80.16%" becomes 0.8016). **HTML (Market Share wiki page)**

4. Visually, we noticed that The flight CSV identifies airlines by two-letter carrier codes (e.g. "DL") while the market share table uses full names (e.g. "Delta Air Lines"), and this is significant because the information in the two datasets cannot be compared unless the airlines are identified the same way. We used unique values from the CSV's "Carrier Code" column to the converted airline names in the scraped table by converting the full airline names to carrier codes to match the CSV, and we also changed the American Eagle code ("MQ") to American Airlines ("AA") in the CSV to match how the market share table groups them. **CSV / Downloaded File & HTML (Market Share wiki page)**

5. (if applicable)
